In [17]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

#Determining parameters
parameters = {
    'J_tot': 1.0,    
    'mu_r': 0.1,      
    'mu_lx': 0.05,    
    'mu_ly': 0.05,    
    'r_m': 0.05,      
    'K_m': 0.1,       
    'L_m': 0.01,      
    'R_m': 1.0,       
    'C_tn': 0.001,    
    'R_tn': 10.0,     
    'C_n': 0.0001,    
    'R_n': 5.0,       
    'L_n': 0.05,      
    'T_c': 20.0       
}

#Gemini computed the matrices from a picture from our report
def get_state_space_matrices(p):

    A = np.zeros((9, 9))
    
    A[0, 1] = 1.0
    A[1, 1] = -(p['mu_r'] / p['J_tot'] + (p['mu_lx'] * p['r_m']**2) / (2 * p['J_tot']))
    A[1, 2] = (p['K_m'] * p['r_m']) / p['J_tot']
    A[2, 1] = -p['K_m'] / (p['L_m'] * p['r_m'])
    A[2, 2] = -p['R_m'] / p['L_m']
    
    A[3, 4] = 1.0
    A[4, 4] = -(p['mu_r'] / p['J_tot'] + (p['mu_ly'] * p['r_m']**2) / (2 * p['J_tot']))
    A[4, 5] = (p['K_m'] * p['r_m']) / p['J_tot']
    A[5, 4] = -p['K_m'] / (p['L_m'] * p['r_m'])
    A[5, 5] = -p['R_m'] / p['L_m']
    
    A[6, 6] = -1.0 / (p['C_tn'] * p['R_tn'])
    A[6, 7] = 1.0 / (p['C_tn'] * p['R_n'])
    A[7, 7] = -1.0 / (p['C_n'] * p['R_n'])
    A[7, 8] = 1.0 / p['C_n']
    A[8, 7] = -1.0 / p['L_n']
    
    B = np.zeros((9, 3))
    B[2, 0] = 1.0 / p['L_m']
    B[5, 1] = 1.0 / p['L_m']
    B[8, 2] = 1.0 / p['L_n']
    
    E = np.zeros((9, 1))
    E[6, 0] = p['T_c'] / (p['C_tn'] * p['R_tn'])
    
    M = np.array([
        [0.5,  0.5],
        [0.5, -0.5]
    ])
    
    return A, B, E, M

def state_space_system(t, state, u_input, p):
    A,B,E,M = get_state_space_matrices(p)

    x = state.reshape((9, 1))

    u_m = np.array([[u_input[0]], [u_input[1]]])
    u_xy = M @ u_m
    u = np.array([[u_xy[0, 0]], [u_xy[1, 0]], [u_input[2]]])

    dxdt = A @ x + B @ u + E

    return dxdt.flatten()

#solving the state space
x0 = np.zeros(9)             
u_control = [3.0, 3.0, 3.0] 
t_span = (0, 5)              
sol = solve_ivp(state_space_system, t_span, x0, args=(u_control, parameters), t_eval=t_eval)

